## Data Preparation Notebook

### Prerequisites
1. Run `scripts/setup.sql` first to create the Snowflake infrastructure and Git integration

### Dataset Information
This quickstart uses the **DeepPCB** dataset - a public open-source dataset for PCB defect detection.
- **Source:** [Surface-Defect-Detection](https://github.com/Charmve/Surface-Defect-Detection/tree/master/DeepPCB)
- **License:** Please review and comply with the dataset's licensing terms before use.

### How it works
This notebook reads data directly from the **Git repository stage** - no manual upload needed!
The PCB dataset is included in the repo under `data/PCBData/` and accessed via:
```
@PCB_GITHUB_REPO/branches/main/data/PCBData/
```

### Purpose
This notebook processes images and labels from the Git repo, converts them to base64, and creates the training data tables.
 

#### Import necessary libraries. Refer to `environment.yml` for the complete list of dependencies.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import json
import os
import base64

## Get Snowflake Session

In [ ]:
# Get Snowflake Session object
session = get_active_session()
session.sql_simplifier_enabled = True

# Add a query tag to the session
session.query_tag = {"origin":"sf_sit-is", 
                     "name":"distributed_ml_crt_imageanomaly_detection", 
                     "version":{"major":1, "minor":0},
                     "attributes":{"is_quickstart":1, "source":"notebook"}}

# Current Environment Details
print('Connection Established with the following parameters:')
print('User      : {}'.format(session.get_current_user()))
print('Role      : {}'.format(session.get_current_role()))
print('Database  : {}'.format(session.get_current_database()))
print('Schema    : {}'.format(session.get_current_schema()))
print('Warehouse : {}'.format(session.get_current_warehouse()))
print('Schema    : {}'.format(session.get_current_schema()))
print('Warehouse : {}'.format(session.get_current_warehouse()))

### Verify Data in Git Repository Stage
Run the following to confirm the data is accessible from the Git repo:

Fetch the latest from Git and list the PCB data files.

In [ ]:
# Fetch latest from Git repository
session.sql("ALTER GIT REPOSITORY PCB_GITHUB_REPO FETCH").collect()

# List files in the Git repository stage
print("Sample data files in Git repo:")
session.sql("LIST @PCB_GITHUB_REPO/branches/main/data/PCBData/").show()

# Copy data from Git repo to DATA_STAGE for processing
print("\nCopying data from Git repo to DATA_STAGE...")
session.sql("COPY FILES INTO @DATA_STAGE/labels/ FROM @PCB_GITHUB_REPO/branches/main/data/PCBData/ PATTERN='.*_not/.*\\.txt'").collect()
session.sql("COPY FILES INTO @DATA_STAGE/images/ FROM @PCB_GITHUB_REPO/branches/main/data/PCBData/ PATTERN='.*_test\\.jpg'").collect()
print("Data copied to DATA_STAGE!")

# Verify
print("\nLabels in DATA_STAGE:")
session.sql("LIST @DATA_STAGE/labels/").show()
print("\nImages in DATA_STAGE:")
session.sql("LIST @DATA_STAGE/images/").show()



### Download Labels 
Download label files from the Snowflake stage, parse each label text file to extract label data, and then save this data into a DataFrame which is written to a Snowflake table called LABELS_TRAIN.

After successful data upload to the Snowflake Internal Stage, follow the below steps. The label contains the Xmin,Ymin,Xmax and Ymax coorrdinates of the defects and the class of defect that will be used to train this supervised model.

In [ ]:
import os
import base64
import pandas as pd

# Download labels from DATA_STAGE
local_train_dir = "/tmp/labels/"
os.makedirs(local_train_dir, exist_ok=True)

# Download all label files
print("Downloading label files...")
session.file.get("@DATA_STAGE/labels/", local_train_dir)
print(f"Downloaded labels to {local_train_dir}")

path_annot = "/tmp/labels/"

# Initialize a list to hold all the data  
data = []  
  
# Walking through the directory to get all label files  
for path, subdirs, files in os.walk(path_annot):  
    for name in files:  
        if name.endswith('.txt'):  # Filter to include only .txt files  
            full_path = os.path.join(path, name)  
            with open(full_path, 'r') as file:  
                for line in file:  
                    parts = line.strip().split()  
                    if len(parts) == 5:
                        xmin, ymin, xmax, ymax,class_id = parts  
                        data.append({  
                            "filename": name.replace('.txt', ''), 
                            "xmin": float(xmin),  
                            "ymin": float(ymin),  
                            "xmax": float(xmax),  
                            "ymax": float(ymax) ,
                            "class": int(class_id)
                        })  
  
# Create a DataFrame  
trainlabels_df = pd.DataFrame(data)
train_labels = session.create_dataframe(trainlabels_df)

train_labels.write.save_as_table("LABELS_TRAIN", mode="overwrite")

### Download Images 
 Downloads image files from a Snowflake stage, encode them in Base64, merges them with label data that was processed in the last step, and inserts the combined information into a Snowflake table named train_images_labels.

In [ ]:
import os
import base64
import pandas as pd

# Download images from DATA_STAGE
local_train_images_dir = "/tmp/images/train"
os.makedirs(local_train_images_dir, exist_ok=True)

# Download all image files
print("Downloading image files...")
session.file.get("@DATA_STAGE/images/", local_train_images_dir)
print(f"Downloaded images to {local_train_images_dir}")

# Convert downloaded images to Base64
images_data = []
for filename in os.listdir(local_train_images_dir):
    if filename.endswith("_test.jpg"):
        filepath = os.path.join(local_train_images_dir, filename)
        with open(filepath, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
            images_data.append({"Filename": filename.replace(".jpg", ""), "image_data": encoded_string})

train_images_base64_data = images_data
print(f"Processed {len(images_data)} images")

# Convert to DataFrames for inserting into separate tables
train_images_df = pd.DataFrame(train_images_base64_data)

print("trainlabels_df columns:", trainlabels_df.columns)
print("train_images_df columns:", train_images_df.columns)

if 'filename' in trainlabels_df.columns:
    trainlabels_df.rename(columns={'filename': 'Filename'}, inplace=True)
# Update labels DataFrame to include `_test.jpg` suffix for matching
trainlabels_df['Filename'] = trainlabels_df['Filename'] + "_test"

merged_train_df = pd.merge(trainlabels_df, train_images_df, how='inner', on='Filename')



# Create the Snowflake table schema to accommodate both image data and label information
create_train_table_query = """
CREATE OR REPLACE TABLE train_images_labels (
    Filename varchar,
    image_data VARCHAR,
    class INT,
    xmin FLOAT,
    ymin FLOAT,
    xmax FLOAT,
    ymax FLOAT
)
"""
# Execute the table creation query
session.sql(create_train_table_query).collect()

# Function to insert merged image and label data into the specified table
def insert_images_and_labels_into_table(merged_df, table_name):
    for index, row in merged_df.iterrows():
        filename = row['Filename'].replace('_test', '')
        insert_query = f"""
        INSERT INTO {table_name} (Filename, image_data, class, xmin, ymin, xmax, ymax)
        VALUES ('{filename}', '{row['image_data']}', '{row['class']}', {row['xmin']}, {row['ymin']}, {row['xmax']}, {row['ymax']})
        """
        print("")
        session.sql(insert_query).collect()

# Insert merged data into the train table
insert_images_and_labels_into_table(merged_train_df, "train_images_labels")

print("Merged image and label data inserted into table.")


trainlabels_df columns: Index(['filename', 'xmin', 'ymin', 'xmax', 'ymax', 'class'], dtype='object')
train_images_df columns: Index(['Filename', 'image_data'], dtype='object')






In [ ]:
from sklearn.model_selection import train_test_split

df=session.table("TRAIN_IMAGES_LABELS").to_pandas()

train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

session.write_pandas(train_df,"training_data", auto_create_table=True, overwrite=True, quote_identifiers=False)

session.write_pandas(test_df,"test_data", auto_create_table=True, overwrite=True,quote_identifiers=False)


## End of Data Preparation

You can now proceed to run `1_Distributed_Model_Training_Snowflake_Notebooks.ipynb` in Snowflake Notebooks.